# Week 9: CCE Ablation Study

This notebook validates that entropy spikes are a meaningful retrieval trigger:

1. **Same Setup** - Uses the exact same codebase and benchmark as Week 8
2. **CCE-Spike Retrieval** - Retrieve when CCE > 3.0 using query + confused tokens
3. **Ablation Baselines** - Random, Fixed-Interval, Query-Only, No-Retrieval
4. **Statistical Analysis** - Bootstrap CIs for rigorous comparison

---

## Setup

In [3]:
# Cell 1: Install dependencies
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn sentence-transformers

import os
import sys

# Create orchestrator directory structure
os.makedirs('/content/orchestrator/entropy', exist_ok=True)
os.makedirs('/content/orchestrator/evaluation', exist_ok=True)
os.makedirs('/content/orchestrator/retrieval', exist_ok=True)

# Add to Python path
sys.path.insert(0, '/content')

print("Dependencies installed!")
print("Next: Run Cell 2 to upload orchestrator files")


Dependencies installed!
Next: Run Cell 2 to upload orchestrator files


In [4]:
# Cell 2: Upload orchestrator module files
from google.colab import files
import shutil

def upload_to_dir(target_dir):
    """Upload .py files to target directory."""
    os.makedirs(target_dir, exist_ok=True)
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith('.py'):
            dest = f'{target_dir}/{filename}'
            shutil.move(filename, dest)
            print(f"  Saved: {dest}")
    return uploaded

print("="*60)
print("Upload orchestrator files from your local repo")
print("="*60)

print("\n1. Upload ENTROPY files:")
print("   From: packages/python-orchestrator/orchestrator/entropy/")
print("   Files: __init__.py, token_classifier.py")
upload_to_dir('/content/orchestrator/entropy')

print("\n2. Upload EVALUATION files:")
print("   From: packages/python-orchestrator/orchestrator/evaluation/")
print("   Files: __init__.py, benchmark.py, metrics.py, runner.py")
upload_to_dir('/content/orchestrator/evaluation')

print("\n" + "="*60)
print("Verifying uploads...")
print("="*60)
!ls -la /content/orchestrator/entropy/
!ls -la /content/orchestrator/evaluation/


Upload orchestrator files from your local repo

1. Upload ENTROPY files:
   From: packages/python-orchestrator/orchestrator/entropy/
   Files: __init__.py, token_classifier.py


Saving __init__.py to __init__.py
Saving calculator.py to calculator.py
Saving cce_computer.py to cce_computer.py
Saving measurement.py to measurement.py
Saving monitor.py to monitor.py
Saving spike_detector.py to spike_detector.py
Saving token_classifier.py to token_classifier.py
  Saved: /content/orchestrator/entropy/__init__.py
  Saved: /content/orchestrator/entropy/calculator.py
  Saved: /content/orchestrator/entropy/cce_computer.py
  Saved: /content/orchestrator/entropy/measurement.py
  Saved: /content/orchestrator/entropy/monitor.py
  Saved: /content/orchestrator/entropy/spike_detector.py
  Saved: /content/orchestrator/entropy/token_classifier.py

2. Upload EVALUATION files:
   From: packages/python-orchestrator/orchestrator/evaluation/
   Files: __init__.py, benchmark.py, metrics.py, runner.py


Saving __init__.py to __init__.py
Saving benchmark.py to benchmark.py
Saving benchmark_generator.py to benchmark_generator.py
Saving metrics.py to metrics.py
Saving runner.py to runner.py
Saving stats.py to stats.py
  Saved: /content/orchestrator/evaluation/__init__.py
  Saved: /content/orchestrator/evaluation/benchmark.py
  Saved: /content/orchestrator/evaluation/benchmark_generator.py
  Saved: /content/orchestrator/evaluation/metrics.py
  Saved: /content/orchestrator/evaluation/runner.py
  Saved: /content/orchestrator/evaluation/stats.py

Verifying uploads...
total 120
drwxr-xr-x 2 root root  4096 Jan 28 19:42 .
drwxr-xr-x 5 root root  4096 Jan 28 19:41 ..
-rw-r--r-- 1 root root  9942 Jan 28 19:42 calculator.py
-rw-r--r-- 1 root root 10474 Jan 28 19:42 cce_computer.py
-rw-r--r-- 1 root root  6854 Jan 28 19:42 __init__.py
-rw-r--r-- 1 root root 11385 Jan 28 19:42 measurement.py
-rw-r--r-- 1 root root 23969 Jan 28 19:42 monitor.py
-rw-r--r-- 1 root root 17802 Jan 28 19:42 spike_detecto

In [5]:
# Cell 3: Verify imports work
import sys
sys.path.insert(0, '/content')

try:
    from orchestrator.entropy.token_classifier import HybridClassifier, KeywordClassifier
    print("[OK] HybridClassifier imported")
except ImportError as e:
    print(f"[WARN] HybridClassifier: {e}")

try:
    from orchestrator.evaluation.metrics import EvaluationMetrics
    print("[OK] EvaluationMetrics imported")
except ImportError as e:
    print(f"[WARN] EvaluationMetrics: {e}")

print("\nReady to continue!")


[OK] HybridClassifier imported
[OK] EvaluationMetrics imported

Ready to continue!


/content/orchestrator/evaluation/benchmark_generator.py:554: SyntaxWarning: invalid escape sequence '\.'
  pattern = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"


In [6]:
# Cell 4: Base imports
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any
from collections import defaultdict
import json
import time
import gc

sys.path.insert(0, '/content')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import warnings
warnings.filterwarnings('ignore')

print("Base imports complete")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Base imports complete
PyTorch: 2.9.0+cu126
CUDA available: True


In [7]:
# Cell 5: Import evaluation modules (with fallbacks)

# Try importing from installed package
try:
    from orchestrator.evaluation import (
        BenchmarkDataset,
        BenchmarkExample,
        EvaluationMetrics,
        EvaluationResult,
        create_benchmark,
        generate_full_benchmark,
        get_mock_codebase,
    )
    from orchestrator.evaluation.benchmark import (
        Difficulty,
        Category,
    )
    from orchestrator.evaluation.runner import (
        BaselineRunner,
        ExperimentRunner,
        BaselineMethod,
    )
    print("All evaluation modules imported successfully!")

except ImportError as e:
    print(f"Import error: {e}")
    print("Creating minimal fallback classes...")

    # Minimal fallback implementations
    from dataclasses import dataclass
    from typing import List, Optional
    from enum import Enum

    class Difficulty(Enum):
        EASY = "easy"
        MEDIUM = "medium"
        HARD = "hard"

    class Category(Enum):
        API_USAGE = "api_usage"
        IMPLEMENTATION = "implementation"
        DEBUGGING = "debugging"

    @dataclass
    class BenchmarkExample:
        id: str
        category: str
        difficulty: str
        query: str
        ground_truth_files: List[str]
        keywords: List[str]
        ground_truth_answer: str

    @dataclass
    class EvaluationResult:
        example_id: str
        answer_correctness: float = 0.0
        context_recall: float = 0.0
        context_precision: float = 0.0
        keyword_coverage: float = 0.0
        tokens_used: int = 0
        num_retrievals: int = 0
        method: str = ""

    class EvaluationMetrics:
        def __init__(self, embedding_model='all-MiniLM-L6-v2'):
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(embedding_model)

        def evaluate(self, example_id, generated_answer, ground_truth_answer,
                     retrieved_files, ground_truth_files, ground_truth_keywords,
                     tokens_used, baseline_tokens, num_retrievals, method):
            from sklearn.metrics.pairwise import cosine_similarity

            # Answer correctness via embedding similarity
            gen_emb = self.model.encode([generated_answer])
            gt_emb = self.model.encode([ground_truth_answer])
            answer_sim = float(cosine_similarity(gen_emb, gt_emb)[0][0])

            # Retrieval recall
            gt_set = set(f.split('/')[-1].lower() for f in ground_truth_files)
            ret_set = set(f.split('/')[-1].lower() for f in retrieved_files)
            recall = len(gt_set & ret_set) / len(gt_set) if gt_set else 0
            precision = len(gt_set & ret_set) / len(ret_set) if ret_set else 0

            # Keyword coverage
            answer_lower = generated_answer.lower()
            kw_hits = sum(1 for kw in ground_truth_keywords if kw.lower() in answer_lower)
            kw_coverage = kw_hits / len(ground_truth_keywords) if ground_truth_keywords else 0

            return EvaluationResult(
                example_id=example_id,
                answer_correctness=answer_sim,
                context_recall=recall,
                context_precision=precision,
                keyword_coverage=kw_coverage,
                tokens_used=tokens_used,
                num_retrievals=num_retrievals,
                method=method
            )

    print("Fallback classes created!")


All evaluation modules imported successfully!


In [8]:
# Cell 10: Initialize metrics
metrics = EvaluationMetrics(embedding_model='all-MiniLM-L6-v2')
print("EvaluationMetrics initialized with improved hallucination detection")

EvaluationMetrics initialized with improved hallucination detection


In [9]:
# Cell 12: Embedding Retriever
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class EmbeddingRetriever:
    def __init__(self, documents: Dict[str, str]):
        self.documents = documents
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.doc_names = list(documents.keys())
        self.doc_contents = list(documents.values())
        self.doc_embeddings = self.model.encode(self.doc_contents)
        print(f"EmbeddingRetriever: {len(documents)} documents indexed")

    def retrieve(self, query: str, top_k: int = 2, deduplicate: bool = True) -> List[Dict]:
        query_emb = self.model.encode([query])
        sims = cosine_similarity(query_emb, self.doc_embeddings)[0]
        top_idx = np.argsort(sims)[-top_k:][::-1]
        return [{'source': self.doc_names[i], 'content': self.doc_contents[i], 'score': float(sims[i])} for i in top_idx]


print("EmbeddingRetriever class defined")

EmbeddingRetriever class defined


In [10]:
# Cell 14: Token Classification & CCE Implementation with Query Evolution
#
# QUERY EVOLUTION STRATEGY:
# - Hop 1: Original query + confused tokens (WHERE you want to go + WHAT you need)
# - Hop 2+: Recent code identifiers + confused tokens (WHERE you are + WHAT you need)

from scipy.stats import entropy as scipy_entropy
from dataclasses import dataclass, field
from typing import Tuple, List, Dict, Any
import random
import re
import os

# Import HybridClassifier with fallback
try:
    from orchestrator.entropy.token_classifier import HybridClassifier, KeywordClassifier
    CLASSIFIER_AVAILABLE = True
    print("HybridClassifier imported successfully")
except ImportError:
    CLASSIFIER_AVAILABLE = False
    print("WARNING: HybridClassifier not available, using inline KeywordClassifier")

    # Inline KeywordClassifier fallback
    CODE_KEYWORDS = {
        'if', 'else', 'elif', 'for', 'while', 'break', 'continue', 'pass',
        'return', 'yield', 'raise', 'try', 'except', 'finally', 'with', 'as',
        'def', 'class', 'lambda', 'async', 'await', 'and', 'or', 'not', 'in', 'is',
        'None', 'True', 'False', 'import', 'from', 'print', 'len', 'range',
        'function', 'const', 'let', 'var', 'switch', 'case', 'default',
        'numpy', 'pandas', 'torch', 'tensorflow', 'sklearn', 'requests',
        'FastAPI', 'Flask', 'Django', 'React', 'useState', 'useEffect',
        'firebase', 'auth', 'database', 'pytest', 'unittest', 'test',
    }

    LANGUAGE_WORDS = {
        'what', 'how', 'why', 'when', 'where', 'which', 'who', 'whom',
        'explain', 'show', 'tell', 'help', 'the', 'a', 'an', 'is', 'are',
        'was', 'were', 'be', 'been', 'have', 'has', 'had', 'do', 'does',
        'will', 'would', 'should', 'can', 'could', 'may', 'might',
    }

    class KeywordClassifier:
        def __init__(self):
            self.code_keywords = {k.lower() for k in CODE_KEYWORDS}
            self.language_words = {w.lower() for w in LANGUAGE_WORDS}

        def classify(self, token):
            token_lower = token.strip().lower()
            if token_lower in self.code_keywords:
                return 'code'
            elif token_lower in self.language_words:
                return 'language'
            return 'other'

    class HybridClassifier(KeywordClassifier):
        """Fallback: just uses keyword classification"""
        def __init__(self, **kwargs):
            super().__init__()
            print("Using KeywordClassifier fallback (no embeddings)")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

@dataclass
class MultiHopRetrievalResult:
    retrieved_files: List[str]
    retrieved_content: str
    scores: List[float]
    num_hops: int
    total_tokens: int
    trace: List[Dict[str, Any]]


def extract_code_identifiers(text: str, classifier) -> List[str]:
    """Extract code-classified identifiers from generated text.

    Uses HybridClassifier to filter out language/other tokens,
    keeping only code-related identifiers for query evolution.

    Args:
        text: Generated text to extract identifiers from
        classifier: HybridClassifier instance

    Returns:
        List of code-classified identifier strings
    """
    # Extract potential identifiers (alphanumeric with underscores)
    candidates = re.findall(r'\b[A-Za-z_][A-Za-z0-9_]{2,}\b', text)

    # Filter to keep only code-classified tokens
    code_identifiers = []
    seen = set()
    for candidate in candidates:
        if candidate.lower() in seen:
            continue
        seen.add(candidate.lower())

        # Classify the token
        try:
            result = classifier.classify(candidate)
            if result == 'code':
                code_identifiers.append(candidate)
        except:
            continue

    return code_identifiers


class CCEQueryPlusTopKRetriever:
    """CCE Multi-Hop with Query Evolution.

    Uses HybridClassifier for token classification and evolves
    queries based on generation state:

    - Hop 1: original_query + confused_tokens (initial intent + immediate need)
    - Hop 2+: recent_identifiers + confused_tokens (current context + immediate need)

    This allows retrieval to follow the model's discoveries rather than
    staying anchored to the original query.
    """

    def __init__(self, base_retriever, tokenizer, model,
                 top_k: int = 2, max_retrievals: int = 5,
                 uncertainty_threshold: float = 3.0,
                 top_k_tokens: int = 10,
                 max_gen_tokens: int = 200,
                 cooldown_tokens: int = 5,
                 file_list_context: str = "",
                 use_hybrid_classifier: bool = True,
                 recent_window_chars: int = 200,
                 max_confused_tokens: int = 5,
                 max_recent_identifiers: int = 5):
        self.retriever = base_retriever
        self.tokenizer = tokenizer
        self.model = model
        self.top_k = top_k
        self.max_retrievals = max_retrievals
        self.uncertainty_threshold = uncertainty_threshold
        self.top_k_tokens = top_k_tokens
        self.max_gen_tokens = max_gen_tokens
        self.cooldown_tokens = cooldown_tokens
        self.file_list_context = file_list_context
        self.use_hybrid_classifier = use_hybrid_classifier

        # Query evolution parameters
        self.recent_window_chars = recent_window_chars  # ~50 tokens
        self.max_confused_tokens = max_confused_tokens
        self.max_recent_identifiers = max_recent_identifiers

        # State tracking for query evolution
        self.hop_count = 0
        self.generated_text = ""

        # Initialize classifier
        if use_hybrid_classifier:
            print("Using HybridClassifier (keyword + embedding fallback)")
            self.classifier = HybridClassifier(
                embedding_model='all-MiniLM-L6-v2',
                embedding_margin=0.05,
                use_embedding_cache=True
            )
        else:
            print("Using KeywordClassifier (keyword-only)")
            self.classifier = KeywordClassifier()

        # Build vocabulary classification (one-time)
        self._build_vocab_classification()

    def reset(self):
        """Reset state for a new query. Call before each retrieve()."""
        self.hop_count = 0
        self.generated_text = ""

    def _build_vocab_classification(self):
        """Classify all tokens in vocabulary using HybridClassifier.

        CACHING: Saves/loads classification to avoid re-running every time.
        Cache file: vocab_cache_{model_name}_{vocab_size}.npz
        """
        import hashlib

        vocab_size = len(self.tokenizer)

        # Generate cache filename based on model
        model_name = getattr(self.tokenizer, 'name_or_path', 'unknown').replace('/', '_')
        cache_file = f"vocab_cache_{model_name}_{vocab_size}.npz"

        # Try to load from cache
        if os.path.exists(cache_file):
            print(f"Loading vocab classification from cache: {cache_file}")
            cached = np.load(cache_file)
            self.code_indices = cached['code_indices']
            self.language_indices = cached['language_indices']
            print(f"  Loaded: {len(self.code_indices)} code, {len(self.language_indices)} language tokens")
            return

        # No cache - do full classification
        self.code_indices = []
        self.language_indices = []
        self.other_indices = []

        print(f"Classifying {vocab_size} tokens (one-time, will be cached)...")
        for token_id in range(vocab_size):
            try:
                token = self.tokenizer.decode([token_id]).strip()
                if not token:
                    self.other_indices.append(token_id)
                    continue

                result = self.classifier.classify(token)

                if result == 'code':
                    self.code_indices.append(token_id)
                elif result == 'language':
                    self.language_indices.append(token_id)
                else:
                    self.other_indices.append(token_id)
            except:
                self.other_indices.append(token_id)

        self.code_indices = np.array(self.code_indices)
        self.language_indices = np.array(self.language_indices)

        # Save to cache
        np.savez(cache_file,
                 code_indices=self.code_indices,
                 language_indices=self.language_indices)
        print(f"Saved vocab classification to: {cache_file}")

        print(f"Vocab classification complete:")
        print(f"  Code tokens: {len(self.code_indices)}")
        print(f"  Language tokens: {len(self.language_indices)}")
        print(f"  Other tokens: {len(self.other_indices)}")

        if self.use_hybrid_classifier and hasattr(self.classifier, 'get_stats'):
            stats = self.classifier.get_stats()
            total = stats['keyword_hits'] + stats['embedding_hits'] + stats['other']
            if total > 0:
                print(f"  Keyword hits: {stats['keyword_hits']} ({100*stats['keyword_hits']/total:.1f}%)")
                print(f"  Embedding hits: {stats['embedding_hits']} ({100*stats['embedding_hits']/total:.1f}%)")

    def _compute_cce(self, logits: torch.Tensor) -> Tuple[float, float, float]:
        logits_np = logits.cpu().numpy()

        if len(self.code_indices) > 0:
            code_logits = logits_np[self.code_indices]
            code_logits_stable = code_logits - np.max(code_logits)
            code_probs = np.exp(code_logits_stable) / np.sum(np.exp(code_logits_stable))
            h_code = float(scipy_entropy(code_probs, base=2))
        else:
            h_code = 0.0

        if len(self.language_indices) > 0:
            lang_logits = logits_np[self.language_indices]
            lang_logits_stable = lang_logits - np.max(lang_logits)
            lang_probs = np.exp(lang_logits_stable) / np.sum(np.exp(lang_logits_stable))
            h_lang = float(scipy_entropy(lang_probs, base=2))
        else:
            h_lang = 0.0

        return h_code - h_lang, h_code, h_lang

    def _extract_code_tokens_from_logits(self, logits: torch.Tensor) -> List[str]:
        """Extract confused code tokens from logits (top-k by probability)."""
        logits_np = logits.cpu().numpy()
        code_logits = logits_np[self.code_indices]
        top_within_code = np.argsort(code_logits)[-self.top_k_tokens:][::-1]

        code_tokens = []
        for i in top_within_code:
            token_id = self.code_indices[i]
            token = self.tokenizer.decode([token_id]).strip()
            if len(token) > 1:
                code_tokens.append(token)
        return code_tokens

    def _build_evolved_query(self, original_query: str, confused_tokens: List[str]) -> str:
        """Build retrieval query that evolves with generation state.

        Hop 1: Original query + confused tokens
        Hop 2+: Recent code identifiers + confused tokens

        Args:
            original_query: The original user query
            confused_tokens: Top-k confused code tokens from current logits

        Returns:
            Evolved query string for retrieval
        """
        self.hop_count += 1

        # Limit confused tokens
        limited_confused = confused_tokens[:self.max_confused_tokens]

        if self.hop_count == 1:
            # First hop: Original intent + immediate confusion
            query = f"{original_query} {' '.join(limited_confused)}"
            print(f"      Query (hop 1): original + confused")
        else:
            # Later hops: Where we ARE now + current confusion
            recent_text = self.generated_text[-self.recent_window_chars:]
            recent_identifiers = extract_code_identifiers(recent_text, self.classifier)

            # Combine recent context with current confusion
            limited_recent = recent_identifiers[-self.max_recent_identifiers:]
            query_parts = limited_recent + limited_confused
            query = ' '.join(query_parts)

            print(f"      Query (hop {self.hop_count}): recent_ids={limited_recent} + confused={limited_confused}")

        return query

    def retrieve(self, query: str) -> MultiHopRetrievalResult:
        # Reset state for new query
        self.reset()

        if self.file_list_context:
            prompt = f"{query}\n\n{self.file_list_context}\n\n"
        else:
            prompt = f"{query}\n\n"

        retrieved_files = []
        retrieved_content = []
        all_scores = []
        trace = []
        seen_files = set()
        retrieval_count = 0
        last_retrieval_pos = -100

        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)
        generated_ids = inputs['input_ids']

        for i in range(self.max_gen_tokens):
            with torch.no_grad():
                outputs = self.model(generated_ids)
                logits = outputs.logits[0, -1, :]

            del outputs  # Free memory
            cce, h_code, h_lang = self._compute_cce(logits)

            if i < 3:
                print(f"    Token {i}: CCE={cce:.3f} (H_code={h_code:.2f}, H_lang={h_lang:.2f})")

            tokens_since_last = i - last_retrieval_pos
            in_cooldown = tokens_since_last < self.cooldown_tokens

            if cce > self.uncertainty_threshold and not in_cooldown and retrieval_count < self.max_retrievals:
                # Extract confused code tokens
                confused_tokens = self._extract_code_tokens_from_logits(logits)

                # Build EVOLVED query based on hop number
                retrieval_query = self._build_evolved_query(query, confused_tokens)

                results = self.retriever.retrieve(retrieval_query, top_k=self.top_k, deduplicate=False)
                new_files = [r for r in results if r['source'] not in seen_files]

                if new_files:
                    for r in new_files:
                        seen_files.add(r['source'])
                        retrieved_files.append(r['source'])
                        retrieved_content.append(r['content'])
                        all_scores.append(r['score'])

                    new_context = "\n\n".join([r['content'] for r in new_files])
                    context_text = f"\n\nRelevant context:\n{new_context}\n\n"
                    context_ids = self.tokenizer.encode(context_text, return_tensors='pt').to(self.model.device)
                    generated_ids = torch.cat([generated_ids, context_ids], dim=-1)

                    # Track the injected context in generated text
                    self.generated_text += context_text

                trace.append({
                    'hop': self.hop_count,
                    'position': i,
                    'cce': cce,
                    'confused_tokens': confused_tokens[:5],
                    'retrieval_query': retrieval_query[:100],
                    'new_files': [r['source'] for r in new_files] if new_files else [],
                })

                retrieval_count += 1
                last_retrieval_pos = i
                print(f"    SPIKE {retrieval_count} at {i}: CCE={cce:.2f}")

            # Generate next token
            next_token = torch.argmax(logits).unsqueeze(0).unsqueeze(0)
            generated_ids = torch.cat([generated_ids, next_token.to(self.model.device)], dim=-1)

            # Track generated text for query evolution
            token_text = self.tokenizer.decode([next_token.item()])
            self.generated_text += token_text

            # Periodic memory cleanup
            if i % 50 == 0 and i > 0:
                torch.cuda.empty_cache()

            if next_token.item() == self.tokenizer.eos_token_id:
                break

        if not trace:
            trace.append({'hop': 0, 'method': 'no_spike_detected'})

        content = "\n\n".join(retrieved_content)
        print(f"    Total: {retrieval_count} retrievals, files: {retrieved_files}")

        return MultiHopRetrievalResult(
            retrieved_files=retrieved_files,
            retrieved_content=content,
            scores=all_scores,
            num_hops=retrieval_count,
            total_tokens=len(self.tokenizer.encode(content)) if content else 0,
            trace=trace
        )

print("CCEQueryPlusTopKRetriever defined with QUERY EVOLUTION:")
print("  - Hop 1: original_query + confused_tokens")
print("  - Hop 2+: recent_code_identifiers + confused_tokens")
print("  - Recent window: 200 chars (~50 tokens)")
print("  - Uses HybridClassifier for identifier filtering")


HybridClassifier imported successfully
CCEQueryPlusTopKRetriever defined with QUERY EVOLUTION:
  - Hop 1: original_query + confused_tokens
  - Hop 2+: recent_code_identifiers + confused_tokens
  - Recent window: 200 chars (~50 tokens)
  - Uses HybridClassifier for identifier filtering


In [11]:
# Cell 13: Load LLM model (IMPROVED - Larger Model Options)

# === MODEL SELECTION ===
# Option 1: Qwen2.5-Coder-1.5B (recommended - 3x larger than 0.5B)
# Option 2: Qwen2.5-Coder-3B (if you have enough VRAM)
# Option 3: CodeLlama-7B-Instruct with 4-bit quantization

MODEL_OPTION = 1  # Change this to try different models

if MODEL_OPTION == 1:
    MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
    USE_QUANTIZATION = False
elif MODEL_OPTION == 2:
    MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
    USE_QUANTIZATION = False
elif MODEL_OPTION == 3:
    MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
    USE_QUANTIZATION = True  # 4-bit to fit in memory
else:
    MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
    USE_QUANTIZATION = False

print(f"Loading {MODEL_NAME}...")
print(f"Quantization: {USE_QUANTIZATION}")

if USE_QUANTIZATION:
    from transformers import BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {MODEL_NAME}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


Loading Qwen/Qwen2.5-Coder-1.5B-Instruct...
Quantization: False


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded: Qwen/Qwen2.5-Coder-1.5B-Instruct
Vocab size: 151665
Model parameters: 1,543,714,304


In [12]:
# Cell 15: Ablation Baselines (Memory-Optimized)

class RandomRetrievalBaseline:
    """Random retrieval baseline - uses upfront retrieval for efficiency."""
    def __init__(self, retriever, tokenizer, model, max_gen_tokens=100, file_list_context=""):
        self.retriever = retriever
        self.tokenizer = tokenizer
        self.model = model
        self.max_gen_tokens = max_gen_tokens
        self.file_list_context = file_list_context

    def retrieve(self, query: str, num_retrievals: int = 2, seed: int = None) -> MultiHopRetrievalResult:
        if seed is not None:
            random.seed(seed)

        results = self.retriever.retrieve(query, top_k=5)

        if len(results) > num_retrievals:
            selected = random.sample(results, num_retrievals)
        else:
            selected = results

        retrieved_files = [r['source'] for r in selected]
        retrieved_content = "\n\n".join([r['content'] for r in selected])
        scores = [r['score'] for r in selected]

        trace = [{'position': random.randint(10, self.max_gen_tokens-10), 'method': 'random'}
                 for _ in range(len(selected))]

        tokens_used = len(self.tokenizer.encode(retrieved_content)) if retrieved_content else 0

        return MultiHopRetrievalResult(
            retrieved_files, retrieved_content, scores,
            len(selected), tokens_used, trace
        )


class FixedIntervalBaseline:
    """Fixed interval baseline - uses upfront retrieval for efficiency."""
    def __init__(self, retriever, tokenizer, model, max_gen_tokens=100, file_list_context=""):
        self.retriever = retriever
        self.tokenizer = tokenizer
        self.model = model
        self.max_gen_tokens = max_gen_tokens
        self.file_list_context = file_list_context

    def retrieve(self, query: str, interval: int = 50, max_retrievals: int = 2) -> MultiHopRetrievalResult:
        results = self.retriever.retrieve(query, top_k=max_retrievals * 2)
        selected = results[:max_retrievals]

        retrieved_files = [r['source'] for r in selected]
        retrieved_content = "\n\n".join([r['content'] for r in selected])
        scores = [r['score'] for r in selected]

        trace = [{'position': (i+1) * interval, 'method': 'fixed_interval'}
                 for i in range(len(selected))]

        tokens_used = len(self.tokenizer.encode(retrieved_content)) if retrieved_content else 0

        return MultiHopRetrievalResult(
            retrieved_files, retrieved_content, scores,
            len(selected), tokens_used, trace
        )


class QueryOnlyBaseline:
    """Query-only baseline."""
    def __init__(self, retriever, tokenizer, model, max_gen_tokens=100):
        self.retriever = retriever
        self.tokenizer = tokenizer
        self.model = model
        self.max_gen_tokens = max_gen_tokens

    def generate(self, query: str) -> Tuple[str, List[str]]:
        results = self.retriever.retrieve(query, top_k=2)
        context = "\n\n".join([r['content'] for r in results])
        files = [r['source'] for r in results]

        prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
        inputs = self.tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                inputs['input_ids'],
                max_new_tokens=self.max_gen_tokens,
                pad_token_id=self.tokenizer.eos_token_id,
                do_sample=False
            )
        answer = self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        del outputs, inputs
        torch.cuda.empty_cache()

        return answer, files


class NoRetrievalBaseline:
    """No retrieval baseline."""
    def __init__(self, tokenizer, model, max_gen_tokens=100):
        self.tokenizer = tokenizer
        self.model = model
        self.max_gen_tokens = max_gen_tokens

    def generate(self, query: str) -> str:
        prompt = f"Question: {query}\nAnswer:"
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                inputs['input_ids'],
                max_new_tokens=self.max_gen_tokens,
                pad_token_id=self.tokenizer.eos_token_id,
                do_sample=False
            )
        answer = self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        del outputs, inputs
        torch.cuda.empty_cache()

        return answer

print("Memory-optimized baselines defined")


Memory-optimized baselines defined


In [13]:
# Cell 11: Define CCE Trace Function (uses masks from Cell 16)

# NOTE: This cell only DEFINES the function.
# The actual token masks (code_token_mask, lang_token_mask) are created in Cell 16
# when we initialize the CCE retriever.

def generate_with_cce_trace(query: str, max_tokens: int = 100) -> Dict:
    """Generate answer WITHOUT retrieval, logging CCE at each token position.

    Uses global code_token_mask and lang_token_mask created in Cell 16.
    """

    prompt = "Question: " + query + "\nAnswer:"
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(model.device)

    tokens_generated = []
    cce_trace = []
    h_code_trace = []
    h_lang_trace = []

    current_ids = input_ids

    for step in range(max_tokens):
        with torch.no_grad():
            outputs = model(current_ids)
            logits = outputs.logits[0, -1, :].float()

            # Verify shapes match
            vocab_size_model = logits.shape[0]

            probs = torch.softmax(logits, dim=-1).cpu().numpy()

            del outputs, logits

            # Use global masks (must be created in Cell 16 first!)
            # Safety check for size mismatch
            if len(probs) != len(code_token_mask):
                print(f"WARNING: Vocab mismatch! probs={len(probs)}, mask={len(code_token_mask)}")
                # Fallback: use simple entropy
                h_code = scipy_entropy(probs + 1e-10, base=2)
                h_lang = h_code * 0.7  # Rough estimate
            else:
                code_probs = probs[code_token_mask]
                lang_probs = probs[lang_token_mask]

                if code_probs.sum() > 1e-10:
                    code_probs_norm = code_probs / code_probs.sum()
                    h_code = scipy_entropy(code_probs_norm + 1e-10, base=2)
                else:
                    h_code = 0.0

                if lang_probs.sum() > 1e-10:
                    lang_probs_norm = lang_probs / lang_probs.sum()
                    h_lang = scipy_entropy(lang_probs_norm + 1e-10, base=2)
                else:
                    h_lang = 0.0

            cce = h_code - h_lang

            cce_trace.append(float(cce))
            h_code_trace.append(float(h_code))
            h_lang_trace.append(float(h_lang))

            next_token_id = int(np.argmax(probs))
            next_token = tokenizer.decode([next_token_id])
            tokens_generated.append(next_token)

            del probs

            if next_token_id == tokenizer.eos_token_id:
                break

            current_ids = torch.cat([current_ids, torch.tensor([[next_token_id]], device=model.device)], dim=1)

            if step % 25 == 0 and step > 0:
                gc.collect()
                torch.cuda.empty_cache()

    del current_ids, input_ids
    gc.collect()
    torch.cuda.empty_cache()

    # Use the SELECTED_THRESHOLD from Cell 16
    threshold = SELECTED_THRESHOLD if 'SELECTED_THRESHOLD' in dir() else 2.5

    return {
        'query': query,
        'answer': ''.join(tokens_generated),
        'tokens': tokens_generated,
        'cce_trace': cce_trace,
        'h_code_trace': h_code_trace,
        'h_lang_trace': h_lang_trace,
        'spike_positions': [i for i, cce in enumerate(cce_trace) if cce > threshold],
    }

print("generate_with_cce_trace function defined")
print("NOTE: Run Cell 16 first to create token masks before using this function!")


generate_with_cce_trace function defined
NOTE: Run Cell 16 first to create token masks before using this function!


In [14]:
# Cell 12: Compute hallucination trace function

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

hallu_embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def compute_hallucination_trace(tokens: List[str], ground_truth: str) -> List[Dict]:
    """
    Compute hallucination score at each token position using semantic distance.

    As generation progresses toward ground truth, similarity should INCREASE.
    A DROP in similarity indicates hallucination.
    """
    gt_embedding = hallu_embed_model.encode([ground_truth])[0]

    trace = []
    cumulative_text = ""
    prev_similarity = 0.0

    for i, token in enumerate(tokens):
        cumulative_text += token

        # Embed cumulative generated text
        gen_embedding = hallu_embed_model.encode([cumulative_text])[0]

        # Compute similarity to ground truth
        similarity = float(cosine_similarity([gen_embedding], [gt_embedding])[0][0])

        # Hallucination score = 1 - similarity (higher = more hallucinated)
        hallucination_score = 1.0 - similarity

        # Detect similarity drop (potential hallucination point)
        similarity_drop = prev_similarity - similarity if i > 0 else 0.0

        trace.append({
            'position': i,
            'token': token,
            'cumulative_similarity': similarity,
            'hallucination_score': hallucination_score,
            'similarity_drop': similarity_drop,
            'is_drop': similarity_drop > 0.01,
        })

        prev_similarity = similarity

    return trace

print("compute_hallucination_trace function defined")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

compute_hallucination_trace function defined


---
## Cerberus Experiment: Real Codebase Validation

Using **Cerberus** (Python validation library) as a real-world codebase to test CCE spike-error correlation.

In [15]:
# Cell A: Clone and Load Codebase (IMPROVED - More Novel Options)

import os
import subprocess

# === CODEBASE SELECTION ===
# Option 1: Cerberus (validation library - might be in training data)
# Option 2: httpx (modern async HTTP client - newer, less common)
# Option 3: Typer (CLI framework - newer)

CODEBASE_OPTION = 2  # Change this to try different codebases

CODEBASE_CONFIG = {
    1: {
        'name': 'cerberus',
        'url': 'https://github.com/pyeve/cerberus.git',
        'src_path': 'cerberus/cerberus',
        'description': 'Python validation library'
    },
    2: {
        'name': 'httpx',
        'url': 'https://github.com/encode/httpx.git',
        'src_path': 'httpx/httpx',
        'description': 'Modern async HTTP client (newer, less common)'
    },
    3: {
        'name': 'typer',
        'url': 'https://github.com/tiangolo/typer.git',
        'src_path': 'typer/typer',
        'description': 'CLI framework by FastAPI creator'
    }
}

config = CODEBASE_CONFIG[CODEBASE_OPTION]
REPO_NAME = config['name']
REPO_URL = config['url']
SRC_PATH = config['src_path']

print("="*70)
print(f"STEP 1: Clone {REPO_NAME.upper()} Repository")
print(f"Description: {config['description']}")
print("="*70)

# Clone if not exists
if not os.path.exists(REPO_NAME):
    subprocess.run(['git', 'clone', REPO_URL, '--depth', '1'], check=True)
    print(f"Cloned {REPO_NAME} repository")
else:
    print(f"{REPO_NAME} already exists, skipping clone")

# Load codebase into dict format
def load_codebase(src_path, repo_name):
    """Load source files into dict format for retriever."""
    files = {}

    for root, dirs, filenames in os.walk(src_path):
        # Skip test directories for cleaner results
        dirs[:] = [d for d in dirs if 'test' not in d.lower()]

        for f in filenames:
            if f.endswith('.py') and not f.startswith('test_'):
                path = os.path.join(root, f)
                rel_path = path.replace(f'{repo_name}/', '')
                try:
                    with open(path, 'r', encoding='utf-8') as fp:
                        content = fp.read()
                        if len(content) > 100:  # Skip tiny files
                            files[rel_path] = content
                except Exception as e:
                    print(f"  Warning: Could not read {path}: {e}")

    return files

target_codebase = load_codebase(SRC_PATH, REPO_NAME)

print(f"\n{REPO_NAME.upper()} Codebase Loaded")
print("="*70)
print(f"Total files: {len(target_codebase)}")
print(f"Total size: {sum(len(v) for v in target_codebase.values()):,} characters")
print("\nFiles:")
for path, content in sorted(target_codebase.items())[:15]:
    print(f"  {path} ({len(content):,} chars)")
if len(target_codebase) > 15:
    print(f"  ... and {len(target_codebase) - 15} more files")

# Create file list context
target_file_list = f"Available files in {REPO_NAME} codebase:\n"
for path in sorted(target_codebase.keys()):
    target_file_list += f"- {path}\n"
target_file_list += f"\nWhen answering questions about {REPO_NAME}, refer to the relevant files above."
print(f"\nFile list context created ({len(target_file_list)} chars)")


STEP 1: Clone HTTPX Repository
Description: Modern async HTTP client (newer, less common)
Cloned httpx repository

HTTPX Codebase Loaded
Total files: 23
Total size: 284,357 characters

Files:
  __init__.py (2,191 chars)
  __version__.py (108 chars)
  _api.py (11,743 chars)
  _auth.py (11,907 chars)
  _client.py (65,713 chars)
  _config.py (8,547 chars)
  _content.py (8,161 chars)
  _decoders.py (12,041 chars)
  _exceptions.py (8,490 chars)
  _main.py (15,626 chars)
  _models.py (44,697 chars)
  _multipart.py (9,843 chars)
  _status_codes.py (5,639 chars)
  _transports/__init__.py (275 chars)
  _transports/asgi.py (5,501 chars)
  ... and 8 more files

File list context created (480 chars)


In [16]:
# Cell B: Dynamic Retrieval Benchmark Examples
#
# IMPORTANT: These benchmarks test MULTI-HOP retrieval where:
# 1. Initial query provides partial information
# 2. Generation discovers new identifiers requiring additional retrieval
# 3. Correct answer requires files retrieved at DIFFERENT hops
#
# This tests the QUERY EVOLUTION strategy where later hops use
# recent identifiers + confused tokens instead of the original query.

from dataclasses import dataclass, field
from typing import List, Dict, Optional
from collections import Counter

@dataclass
class DynamicBenchmarkExample:
    """Benchmark example designed for multi-hop dynamic retrieval."""
    id: str
    category: str
    difficulty: str
    task: str                               # Generation TASK (not Q&A lookup)
    ground_truth_files_ordered: List[str]   # Files in ORDER they should be retrieved
    hop_triggers: List[Dict]                # What triggers each hop
    intermediate_discoveries: List[str]     # What's learned at each hop
    ground_truth_answer: str
    confused_token_hints: List[str]
    requires_all_hops: bool = True
    min_hops_required: int = 2


def score_multi_hop_retrieval(
    retrieved_files: List[str],
    example: DynamicBenchmarkExample
) -> Dict[str, float]:
    """
    Score retrieval against multi-hop ground truth.

    Returns:
        dict with:
        - recall: What % of required files were retrieved
        - precision: What % of retrieved files were relevant
        - order_score: How well retrieval order matches expected
        - hop_coverage: What % of required hops were completed
    """
    ground_truth = set(example.ground_truth_files_ordered)
    retrieved = set(retrieved_files)

    # Basic metrics
    true_positives = ground_truth & retrieved
    recall = len(true_positives) / len(ground_truth) if ground_truth else 0
    precision = len(true_positives) / len(retrieved) if retrieved else 0

    # Order score (how well retrieval order matches expected)
    order_score = 0.0
    if retrieved_files:
        gt_order = {f: i for i, f in enumerate(example.ground_truth_files_ordered)}
        matched_order = []
        for f in retrieved_files:
            # Match by filename (strip path prefixes)
            f_name = f.split('/')[-1]
            for gt_file in gt_order:
                if gt_file.endswith(f_name) or f.endswith(gt_file.split('/')[-1]):
                    matched_order.append(gt_order[gt_file])
                    break

        if len(matched_order) >= 2:
            in_order = sum(1 for i in range(len(matched_order)-1)
                          if matched_order[i] < matched_order[i+1])
            order_score = in_order / (len(matched_order) - 1)
        elif len(matched_order) == 1:
            order_score = 1.0 if matched_order[0] == 0 else 0.5

    # Hop coverage
    hop_coverage = len(true_positives) / example.min_hops_required

    return {
        'recall': recall,
        'precision': precision,
        'f1': 2 * recall * precision / (recall + precision) if (recall + precision) > 0 else 0,
        'order_score': order_score,
        'hop_coverage': min(1.0, hop_coverage),
        'complete': recall == 1.0,
    }


# =============================================================================
# HTTPX DYNAMIC BENCHMARKS (Primary)
# =============================================================================

HTTPX_DYNAMIC_BENCHMARKS = [
    # Implementation Tasks
    DynamicBenchmarkExample(
        id='dyn_impl_001',
        category='implementation',
        difficulty='hard',
        task='Implement a retry mechanism for failed httpx requests with exponential backoff',
        ground_truth_files_ordered=[
            'httpx/_client.py',
            'httpx/_exceptions.py',
            'httpx/_config.py',
        ],
        hop_triggers=[
            {'hop': 1, 'trigger': 'initial_task', 'discovers': ['Client', 'request', 'send']},
            {'hop': 2, 'trigger': 'exception_handling', 'discovers': ['TimeoutException', 'ConnectError']},
            {'hop': 3, 'trigger': 'configuration', 'discovers': ['Timeout', 'Limits']},
        ],
        intermediate_discoveries=[
            'Hop 1: Client.send() is the core request method',
            'Hop 2: TimeoutException and ConnectError are retryable',
            'Hop 3: Configuration follows Timeout/Limits pattern',
        ],
        ground_truth_answer='Wrap Client.send() with retry logic catching TimeoutException/ConnectError, use exponential backoff with Retry config class',
        confused_token_hints=['retry', 'backoff', 'timeout', 'exception', 'attempts'],
        min_hops_required=3,
    ),

    DynamicBenchmarkExample(
        id='dyn_impl_002',
        category='implementation',
        difficulty='hard',
        task='Add request/response logging middleware to httpx client',
        ground_truth_files_ordered=[
            'httpx/_client.py',
            'httpx/_transports/base.py',
            'httpx/_models.py',
        ],
        hop_triggers=[
            {'hop': 1, 'trigger': 'initial_task', 'discovers': ['Client', '_transport', 'send']},
            {'hop': 2, 'trigger': 'transport_discovery', 'discovers': ['BaseTransport', 'handle_request']},
            {'hop': 3, 'trigger': 'model_details', 'discovers': ['Request', 'Response', 'headers']},
        ],
        intermediate_discoveries=[
            'Hop 1: Client uses _transport.handle_request()',
            'Hop 2: BaseTransport has handle_request interface',
            'Hop 3: Request/Response have url, method, headers, status_code',
        ],
        ground_truth_answer='Create LoggingTransport(BaseTransport) wrapper that logs Request/Response details in handle_request',
        confused_token_hints=['transport', 'middleware', 'logging', 'request', 'response'],
        min_hops_required=3,
    ),

    # Debugging Tasks
    DynamicBenchmarkExample(
        id='dyn_debug_001',
        category='debugging',
        difficulty='hard',
        task='Debug why requests timeout when using a proxy with httpx',
        ground_truth_files_ordered=[
            'httpx/_exceptions.py',
            'httpx/_config.py',
            'httpx/_transports/default.py',
        ],
        hop_triggers=[
            {'hop': 1, 'trigger': 'timeout_error', 'discovers': ['TimeoutException', 'ConnectTimeout']},
            {'hop': 2, 'trigger': 'timeout_config', 'discovers': ['Timeout', 'connect', 'read']},
            {'hop': 3, 'trigger': 'proxy_transport', 'discovers': ['HTTPTransport', 'proxy']},
        ],
        intermediate_discoveries=[
            'Hop 1: ConnectTimeout vs ReadTimeout - proxy uses connect phase',
            'Hop 2: Timeout has separate connect/read values',
            'Hop 3: Proxy requires extra connect phase through tunnel',
        ],
        ground_truth_answer='Proxy requires TWO connect phases, increase Timeout(connect=30.0) for proxy chain',
        confused_token_hints=['timeout', 'proxy', 'connect', 'tunnel', 'transport'],
        min_hops_required=3,
    ),

    DynamicBenchmarkExample(
        id='dyn_debug_002',
        category='debugging',
        difficulty='hard',
        task='Debug why httpx async requests are slower than expected',
        ground_truth_files_ordered=[
            'httpx/_client.py',
            'httpx/_transports/default.py',
            'httpx/_config.py',
        ],
        hop_triggers=[
            {'hop': 1, 'trigger': 'async_client', 'discovers': ['AsyncClient', 'aclose', '_transport']},
            {'hop': 2, 'trigger': 'async_transport', 'discovers': ['AsyncHTTPTransport', 'httpcore']},
            {'hop': 3, 'trigger': 'pool_limits', 'discovers': ['Limits', 'max_connections']},
        ],
        intermediate_discoveries=[
            'Hop 1: AsyncClient reuses connections if not closed',
            'Hop 2: AsyncHTTPTransport manages connection pool',
            'Hop 3: Default Limits may bottleneck concurrent requests',
        ],
        ground_truth_answer='Use context manager, increase Limits(max_connections=100), enable HTTP/2 for multiplexing',
        confused_token_hints=['async', 'await', 'pool', 'connections', 'concurrent'],
        min_hops_required=3,
    ),

    # Extension Tasks
    DynamicBenchmarkExample(
        id='dyn_ext_001',
        category='extension',
        difficulty='hard',
        task='Add automatic content decompression for custom encodings in httpx',
        ground_truth_files_ordered=[
            'httpx/_models.py',
            'httpx/_content.py',
            'httpx/_decoders.py',
        ],
        hop_triggers=[
            {'hop': 1, 'trigger': 'response_content', 'discovers': ['Response', 'content', 'stream']},
            {'hop': 2, 'trigger': 'content_encoding', 'discovers': ['encode_content', 'ByteStream']},
            {'hop': 3, 'trigger': 'decoders', 'discovers': ['ContentDecoder', 'GZipDecoder']},
        ],
        intermediate_discoveries=[
            'Hop 1: Response.content uses stream with decoder chain',
            'Hop 2: Content encoding flows through ByteStream',
            'Hop 3: Decoders implement decode(data) interface',
        ],
        ground_truth_answer='Create CustomDecoder(ContentDecoder), register in SUPPORTED_DECODERS, Response auto-selects by Content-Encoding',
        confused_token_hints=['decoder', 'content', 'encoding', 'gzip', 'stream'],
        min_hops_required=3,
    ),

    DynamicBenchmarkExample(
        id='dyn_ext_002',
        category='extension',
        difficulty='expert',
        task='Implement request signing (HMAC) as httpx authentication',
        ground_truth_files_ordered=[
            'httpx/_auth.py',
            'httpx/_models.py',
            'httpx/_client.py',
        ],
        hop_triggers=[
            {'hop': 1, 'trigger': 'auth_interface', 'discovers': ['Auth', 'auth_flow', 'require_request_body']},
            {'hop': 2, 'trigger': 'request_signing', 'discovers': ['Request', 'url', 'method', 'content']},
            {'hop': 3, 'trigger': 'auth_flow', 'discovers': ['Client', '_build_auth', 'request']},
        ],
        intermediate_discoveries=[
            'Hop 1: Auth uses generator-based auth_flow',
            'Hop 2: Request has url, method, headers, content for signing',
            'Hop 3: Client calls auth before sending',
        ],
        ground_truth_answer='Subclass Auth with require_request_body=True, compute HMAC in auth_flow, add Authorization header',
        confused_token_hints=['auth', 'hmac', 'signature', 'header', 'request'],
        min_hops_required=3,
    ),

    # Integration Tasks
    DynamicBenchmarkExample(
        id='dyn_int_001',
        category='integration',
        difficulty='hard',
        task='Integrate httpx with OpenTelemetry for distributed tracing',
        ground_truth_files_ordered=[
            'httpx/_client.py',
            'httpx/_transports/base.py',
            'httpx/_models.py',
        ],
        hop_triggers=[
            {'hop': 1, 'trigger': 'client_lifecycle', 'discovers': ['Client', 'send', '_transport']},
            {'hop': 2, 'trigger': 'transport_hooks', 'discovers': ['BaseTransport', 'handle_request']},
            {'hop': 3, 'trigger': 'header_injection', 'discovers': ['Request', 'headers', 'Headers']},
        ],
        intermediate_discoveries=[
            'Hop 1: Client.send() is interception point',
            'Hop 2: Transport wrapper adds span around handle_request',
            'Hop 3: Request.headers is mutable for trace context',
        ],
        ground_truth_answer='Create TracingTransport wrapper, inject traceparent header, add span attributes for http.method/url/status',
        confused_token_hints=['tracing', 'span', 'headers', 'transport', 'context'],
        min_hops_required=3,
    ),

    # Complex Multi-File Tasks
    DynamicBenchmarkExample(
        id='dyn_int_002',
        category='integration',
        difficulty='expert',
        task='Implement circuit breaker pattern for httpx requests',
        ground_truth_files_ordered=[
            'httpx/_client.py',
            'httpx/_exceptions.py',
            'httpx/_config.py',
            'httpx/_transports/base.py',
        ],
        hop_triggers=[
            {'hop': 1, 'trigger': 'initial_task', 'discovers': ['Client', 'send', 'request']},
            {'hop': 2, 'trigger': 'failure_types', 'discovers': ['ConnectError', 'TimeoutException']},
            {'hop': 3, 'trigger': 'config_pattern', 'discovers': ['Timeout', 'Limits', 'dataclass']},
            {'hop': 4, 'trigger': 'clean_wrapper', 'discovers': ['BaseTransport', 'handle_request']},
        ],
        intermediate_discoveries=[
            'Hop 1: Client.send() calls transport',
            'Hop 2: ConnectError, TimeoutException count as failures',
            'Hop 3: Use dataclass for CircuitBreakerConfig',
            'Hop 4: Transport wrapper is cleanest integration',
        ],
        ground_truth_answer='CircuitBreakerTransport wrapper with CLOSED/OPEN/HALF_OPEN states, track failures per host, raise CircuitOpenError',
        confused_token_hints=['circuit', 'breaker', 'failure', 'threshold', 'transport'],
        min_hops_required=4,
    ),

    DynamicBenchmarkExample(
        id='dyn_complex_001',
        category='complex',
        difficulty='expert',
        task='Implement request caching with ETags and conditional requests for httpx',
        ground_truth_files_ordered=[
            'httpx/_client.py',
            'httpx/_models.py',
            'httpx/_transports/base.py',
            'httpx/_content.py',
        ],
        hop_triggers=[
            {'hop': 1, 'trigger': 'initial_task', 'discovers': ['Client', 'send', 'Response']},
            {'hop': 2, 'trigger': 'etag_headers', 'discovers': ['Response', 'headers', 'ETag']},
            {'hop': 3, 'trigger': 'cache_layer', 'discovers': ['BaseTransport', 'handle_request']},
            {'hop': 4, 'trigger': 'content_storage', 'discovers': ['ByteStream', 'content']},
        ],
        intermediate_discoveries=[
            'Hop 1: Client request flow, Response from transport',
            'Hop 2: Response.headers contains ETag, Last-Modified',
            'Hop 3: Transport wrapper for cache interception',
            'Hop 4: Cache stores response content as bytes',
        ],
        ground_truth_answer='CachingTransport wraps BaseTransport, add If-None-Match header, return cached on 304, store ETag + content',
        confused_token_hints=['cache', 'etag', 'conditional', 'transport', 'headers'],
        min_hops_required=4,
    ),

    # Additional implementation task
    DynamicBenchmarkExample(
        id='dyn_impl_003',
        category='implementation',
        difficulty='expert',
        task='Implement connection pooling with per-host limits for httpx',
        ground_truth_files_ordered=[
            'httpx/_client.py',
            'httpx/_config.py',
            'httpx/_transports/default.py',
        ],
        hop_triggers=[
            {'hop': 1, 'trigger': 'initial_task', 'discovers': ['Client', 'limits', '_transport']},
            {'hop': 2, 'trigger': 'limits_config', 'discovers': ['Limits', 'max_connections']},
            {'hop': 3, 'trigger': 'transport_pool', 'discovers': ['HTTPTransport', 'httpcore', '_pool']},
        ],
        intermediate_discoveries=[
            'Hop 1: Client accepts limits, passes to transport',
            'Hop 2: Limits controls pool size',
            'Hop 3: HTTPTransport wraps httpcore pool',
        ],
        ground_truth_answer='Extend Limits with per_host_max_connections, modify HTTPTransport for per-host pool tracking',
        confused_token_hints=['pool', 'connections', 'limits', 'host', 'transport'],
        min_hops_required=3,
    ),
]


# =============================================================================
# CERBERUS DYNAMIC BENCHMARKS
# =============================================================================

CERBERUS_DYNAMIC_BENCHMARKS = [
    DynamicBenchmarkExample(
        id='dyn_cerb_001',
        category='implementation',
        difficulty='hard',
        task='Implement a custom async validation rule for Cerberus that validates against external API',
        ground_truth_files_ordered=[
            'cerberus/validator.py',
            'cerberus/schema.py',
            'cerberus/errors.py',
        ],
        hop_triggers=[
            {'hop': 1, 'trigger': 'validation_rule', 'discovers': ['Validator', '_validate_']},
            {'hop': 2, 'trigger': 'rule_registry', 'discovers': ['SchemaRegistry', 'rules']},
            {'hop': 3, 'trigger': 'error_handling', 'discovers': ['ValidationError', '_error']},
        ],
        intermediate_discoveries=[
            'Hop 1: Rules are _validate_<name>(self, constraint, field, value)',
            'Hop 2: Schema rules registered in SchemaRegistry',
            'Hop 3: Errors added via _error(field, message)',
        ],
        ground_truth_answer='Create _validate_async_api method, store async results, add _error() for failures',
        confused_token_hints=['validate', 'async', 'rule', 'constraint', 'error'],
        min_hops_required=3,
    ),

    DynamicBenchmarkExample(
        id='dyn_cerb_002',
        category='debugging',
        difficulty='hard',
        task='Debug why nested schema validation fails silently in Cerberus',
        ground_truth_files_ordered=[
            'cerberus/validator.py',
            'cerberus/schema.py',
            'cerberus/errors.py',
        ],
        hop_triggers=[
            {'hop': 1, 'trigger': 'nested_validation', 'discovers': ['_validate_schema', 'child_validator']},
            {'hop': 2, 'trigger': 'schema_compile', 'discovers': ['Schema', 'expand']},
            {'hop': 3, 'trigger': 'nested_errors', 'discovers': ['ErrorTree', '_errors']},
        ],
        intermediate_discoveries=[
            'Hop 1: Nested schemas use child_validator',
            'Hop 2: Schema must be expanded before validation',
            'Hop 3: Nested errors in ErrorTree may not surface',
        ],
        ground_truth_answer='Child validators have separate _errors, use verbose error handler to see nested tree',
        confused_token_hints=['nested', 'schema', 'child', 'errors', 'validate'],
        min_hops_required=3,
    ),
]


# =============================================================================
# TYPER DYNAMIC BENCHMARKS
# =============================================================================

TYPER_DYNAMIC_BENCHMARKS = [
    DynamicBenchmarkExample(
        id='dyn_typer_001',
        category='implementation',
        difficulty='hard',
        task='Implement custom parameter type with validation for Typer CLI',
        ground_truth_files_ordered=[
            'typer/main.py',
            'typer/params.py',
            'typer/core.py',
        ],
        hop_triggers=[
            {'hop': 1, 'trigger': 'parameter_type', 'discovers': ['Typer', 'command', 'Argument']},
            {'hop': 2, 'trigger': 'param_definition', 'discovers': ['Argument', 'Option', 'ParamMeta']},
            {'hop': 3, 'trigger': 'click_type', 'discovers': ['click', 'ParamType', 'convert']},
        ],
        intermediate_discoveries=[
            'Hop 1: Typer wraps Click, parameters via annotations',
            'Hop 2: Argument/Option pass click_type to Click',
            'Hop 3: Custom types need click.ParamType.convert()',
        ],
        ground_truth_answer='Subclass click.ParamType with convert(), use with typer.Argument(click_type=MyParamType())',
        confused_token_hints=['type', 'param', 'convert', 'click', 'validate'],
        min_hops_required=3,
    ),
]


# Select benchmarks based on codebase
if CODEBASE_OPTION == 1:
    dynamic_benchmarks = CERBERUS_DYNAMIC_BENCHMARKS
elif CODEBASE_OPTION == 2:
    dynamic_benchmarks = HTTPX_DYNAMIC_BENCHMARKS
elif CODEBASE_OPTION == 3:
    dynamic_benchmarks = TYPER_DYNAMIC_BENCHMARKS

# Also keep old benchmark format for compatibility (converted)
benchmark_examples = []
for ex in dynamic_benchmarks:
    # Convert to old BenchmarkExample format for backward compat
    benchmark_examples.append(type('BenchmarkExample', (), {
        'id': ex.id,
        'category': ex.category,
        'difficulty': ex.difficulty,
        'query': ex.task,
        'ground_truth_files': ex.ground_truth_files_ordered,
        'keywords': ex.confused_token_hints,
        'ground_truth_answer': ex.ground_truth_answer,
    })())

print("=" * 70)
print(f"{REPO_NAME.upper()} DYNAMIC RETRIEVAL BENCHMARKS")
print("=" * 70)
print(f"Total examples: {len(dynamic_benchmarks)}")
print(f"\nBy Category:")
cat_counts = Counter(ex.category for ex in dynamic_benchmarks)
for cat, count in sorted(cat_counts.items()):
    print(f"  {cat}: {count}")
print(f"\nBy Difficulty:")
diff_counts = Counter(ex.difficulty for ex in dynamic_benchmarks)
for diff, count in sorted(diff_counts.items()):
    print(f"  {diff}: {count}")
print(f"\nBy Min Hops Required:")
hop_counts = Counter(ex.min_hops_required for ex in dynamic_benchmarks)
for hops, count in sorted(hop_counts.items()):
    print(f"  {hops} hops: {count}")

print("\n" + "=" * 70)
print("BENCHMARK EXAMPLES:")
print("=" * 70)
for ex in dynamic_benchmarks[:3]:
    print(f"\n[{ex.id}] {ex.task[:60]}...")
    print(f"  Expected retrieval order: {' -> '.join(f.split('/')[-1] for f in ex.ground_truth_files_ordered)}")
    print(f"  Min hops: {ex.min_hops_required}")


HTTPX DYNAMIC RETRIEVAL BENCHMARKS
Total examples: 10

By Category:
  complex: 1
  debugging: 2
  extension: 2
  implementation: 3
  integration: 2

By Difficulty:
  expert: 4
  hard: 6

By Min Hops Required:
  3 hops: 8
  4 hops: 2

BENCHMARK EXAMPLES:

[dyn_impl_001] Implement a retry mechanism for failed httpx requests with e...
  Expected retrieval order: _client.py -> _exceptions.py -> _config.py
  Min hops: 3

[dyn_impl_002] Add request/response logging middleware to httpx client...
  Expected retrieval order: _client.py -> base.py -> _models.py
  Min hops: 3

[dyn_debug_001] Debug why requests timeout when using a proxy with httpx...
  Expected retrieval order: _exceptions.py -> _config.py -> default.py
  Min hops: 3


In [17]:

# Cell C: Initialize Retrievers (Creates token masks for generate_with_cce_trace)

print("="*70)
print(f"INITIALIZING RETRIEVERS FOR {REPO_NAME.upper()}")
print("="*70)

# Create embedding retriever
target_retriever = EmbeddingRetriever(target_codebase)
print(f"Embedding retriever initialized with {len(target_codebase)} files")

# CCE Threshold
CCE_THRESHOLDS = [2.0, 2.5, 3.0, 3.5, 4.0]
SELECTED_THRESHOLD = 2.5

print(f"\nCCE Threshold: {SELECTED_THRESHOLD}")

# Create CCE retriever
print(f"Initializing CCE Retriever...")
target_cce_retriever = CCEQueryPlusTopKRetriever(
    base_retriever=target_retriever,
    tokenizer=tokenizer,
    model=model,
    uncertainty_threshold=SELECTED_THRESHOLD,
    max_gen_tokens=150,
    file_list_context=target_file_list
)
print("CCE retriever initialized!")

# === CREATE GLOBAL TOKEN MASKS ===
# These are used by generate_with_cce_trace in Cell 18
print("\nCreating global token masks...")

# Get vocab size directly from tokenizer (most reliable)
vocab_size = len(tokenizer)
print(f"Tokenizer vocab size: {vocab_size}")

# Verify with model config
model_vocab = model.config.vocab_size
print(f"Model config vocab size: {model_vocab}")

# Use the larger of the two to be safe
actual_vocab_size = max(vocab_size, model_vocab)
print(f"Using vocab size: {actual_vocab_size}")

# Create masks
code_token_mask = np.zeros(actual_vocab_size, dtype=bool)
lang_token_mask = np.zeros(actual_vocab_size, dtype=bool)

# Fill from CCE retriever's classification
for idx in target_cce_retriever.code_indices:
    if idx < actual_vocab_size:
        code_token_mask[idx] = True

for idx in target_cce_retriever.language_indices:
    if idx < actual_vocab_size:
        lang_token_mask[idx] = True

print(f"Token masks created: {code_token_mask.sum()} code, {lang_token_mask.sum()} language")

# Verify masks work
test_probs = np.random.rand(actual_vocab_size)
try:
    test_code = test_probs[code_token_mask]
    test_lang = test_probs[lang_token_mask]
    print(f"Mask verification: OK (code={len(test_code)}, lang={len(test_lang)})")
except Exception as e:
    print(f"Mask verification FAILED: {e}")

# Create baseline retrievers
target_random_baseline = RandomRetrievalBaseline(
    target_retriever, tokenizer, model, 150, target_file_list
)
target_fixed_baseline = FixedIntervalBaseline(
    target_retriever, tokenizer, model, 150, target_file_list
)

print("\nAll retrievers initialized!")

# Compute baseline tokens
sep = "\n\n"
target_full_context = sep.join([f"# {path}\n{content}" for path, content in list(target_codebase.items())[:10]])
target_baseline_tokens = len(tokenizer.encode(target_full_context[:30000]))
print(f"Baseline tokens (sample): {target_baseline_tokens:,}")


INITIALIZING RETRIEVERS FOR HTTPX
EmbeddingRetriever: 23 documents indexed
Embedding retriever initialized with 23 files

CCE Threshold: 2.5
Initializing CCE Retriever...
Using HybridClassifier (keyword + embedding fallback)
Loading vocab classification from cache: vocab_cache_Qwen_Qwen2.5-Coder-1.5B-Instruct_151665.npz
  Loaded: 43127 code, 12952 language tokens
CCE retriever initialized!

Creating global token masks...
Tokenizer vocab size: 151665
Model config vocab size: 151936
Using vocab size: 151936
Token masks created: 43127 code, 12952 language
Mask verification: OK (code=43127, lang=12952)

All retrievers initialized!
Baseline tokens (sample): 6,539


In [18]:
# Cell D: Run Ablation Study with Multi-Hop Scoring
#
# ENHANCED: Now tracks multi-hop retrieval quality:
# - order_score: Did files get retrieved in expected order?
# - hop_coverage: Did we complete all required hops?
# - per-hop analysis for query evolution validation

NUM_EXAMPLES = 15  # Use all dynamic benchmarks
BATCH_SIZE = 3

print("="*70)
print(f"{REPO_NAME.upper()} ABLATION STUDY (MULTI-HOP SCORING)")
print("="*70)
print(f"Examples: {min(NUM_EXAMPLES, len(dynamic_benchmarks))}")
print(f"CCE Threshold: {SELECTED_THRESHOLD}")
print(f"Model: {MODEL_NAME}")
print("="*70)

def cleanup_memory():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def generate_answer(prompt: str, max_tokens: int = 150) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2000).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    del outputs, inputs
    cleanup_memory()
    return answer

# Sample examples (use dynamic benchmarks)
import random
random.seed(42)

sampled_examples = dynamic_benchmarks[:min(NUM_EXAMPLES, len(dynamic_benchmarks))]
print(f"Using {len(sampled_examples)} dynamic benchmark examples")

# Storage - enhanced with multi-hop metrics
ablation_results = {
    'cce_spike': [],
    'random': [],
    'fixed': [],
    'query_only': [],
    'no_retrieval': []
}

# Multi-hop specific tracking
multihop_stats = {
    'cce_spike': {'order_scores': [], 'hop_coverages': [], 'complete_count': 0},
    'random': {'order_scores': [], 'hop_coverages': [], 'complete_count': 0},
    'fixed': {'order_scores': [], 'hop_coverages': [], 'complete_count': 0},
    'query_only': {'order_scores': [], 'hop_coverages': [], 'complete_count': 0},
}

# Track CCE behavior
cce_stats = {
    'total_spikes': 0,
    'total_retrievals': 0,
    'spike_positions': [],
    'cce_values': [],
    'query_evolution_used': 0,  # Count hops where query evolved
}

def evaluate_multihop(retrieved_files: List[str], example: DynamicBenchmarkExample) -> Dict:
    """Score retrieval using multi-hop ground truth."""
    ground_truth = set(example.ground_truth_files_ordered)
    retrieved = set(retrieved_files)

    # Match files by name (strip path variations)
    def normalize_path(p):
        return p.split('/')[-1].lower()

    gt_normalized = {normalize_path(f): f for f in ground_truth}
    retrieved_normalized = {normalize_path(f): f for f in retrieved}

    # Find matches
    matched = set(gt_normalized.keys()) & set(retrieved_normalized.keys())

    recall = len(matched) / len(ground_truth) if ground_truth else 0
    precision = len(matched) / len(retrieved) if retrieved else 0

    # Order score
    order_score = 0.0
    if retrieved_files and matched:
        gt_order = {normalize_path(f): i for i, f in enumerate(example.ground_truth_files_ordered)}
        matched_order = []
        for f in retrieved_files:
            fn = normalize_path(f)
            if fn in gt_order:
                matched_order.append(gt_order[fn])

        if len(matched_order) >= 2:
            in_order = sum(1 for i in range(len(matched_order)-1)
                          if matched_order[i] < matched_order[i+1])
            order_score = in_order / (len(matched_order) - 1)
        elif len(matched_order) == 1:
            order_score = 1.0 if matched_order[0] == 0 else 0.5

    hop_coverage = min(1.0, len(matched) / example.min_hops_required)

    return {
        'recall': recall,
        'precision': precision,
        'f1': 2 * recall * precision / (recall + precision) if (recall + precision) > 0 else 0,
        'order_score': order_score,
        'hop_coverage': hop_coverage,
        'complete': recall >= 0.99,  # All required files retrieved (with tolerance)
        'matched_files': list(matched),
        'missed_files': list(set(gt_normalized.keys()) - matched),
    }

def evaluate_example(example, answer: str, retrieved_files: list, tokens_used: int,
                     method_name: str, num_retrievals: int = 0):
    return metrics.evaluate(
        example_id=example.id,
        generated_answer=answer,
        ground_truth_answer=example.ground_truth_answer,
        retrieved_files=retrieved_files,
        ground_truth_files=example.ground_truth_files_ordered,
        ground_truth_keywords=example.confused_token_hints,
        tokens_used=tokens_used,
        baseline_tokens=target_baseline_tokens,
        num_retrievals=num_retrievals,
        method=method_name
    )

# Process in batches
num_batches = (len(sampled_examples) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_idx in range(num_batches):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(sampled_examples))
    batch = sampled_examples[start_idx:end_idx]

    print(f"\n{'='*60}")
    print(f"Batch {batch_idx+1}/{num_batches}")
    print(f"{'='*60}")

    for idx, ex in enumerate(batch):
        global_idx = start_idx + idx
        print(f"\n[{global_idx+1}/{len(sampled_examples)}] {ex.id}")
        print(f"  Task: {ex.task[:60]}...")
        print(f"  Expected: {' -> '.join(f.split('/')[-1] for f in ex.ground_truth_files_ordered)}")
        print(f"  Min hops: {ex.min_hops_required}")

        # === CCE-Spike with Query Evolution ===
        cleanup_memory()
        print(f"\n  CCE-Spike:")
        cce_result = target_cce_retriever.retrieve(ex.task)

        cce_stats['total_retrievals'] += cce_result.num_hops

        # Multi-hop scoring
        mh_score = evaluate_multihop(cce_result.retrieved_files, ex)
        multihop_stats['cce_spike']['order_scores'].append(mh_score['order_score'])
        multihop_stats['cce_spike']['hop_coverages'].append(mh_score['hop_coverage'])
        if mh_score['complete']:
            multihop_stats['cce_spike']['complete_count'] += 1

        # Track query evolution
        if cce_result.trace:
            for hop_info in cce_result.trace:
                if 'retrieval_query' in hop_info and hop_info.get('hop', 0) > 1:
                    cce_stats['query_evolution_used'] += 1

        if cce_result.num_hops > 0 and cce_result.retrieved_content:
            prompt = f"Context:\n{cce_result.retrieved_content[:3000]}\n\nTask: {ex.task}\nImplementation:"
        else:
            prompt = f"Task: {ex.task}\nImplementation:"

        cce_answer = generate_answer(prompt)
        cce_eval = evaluate_example(ex, cce_answer, cce_result.retrieved_files,
                                    len(tokenizer.encode(prompt)), 'cce_spike', cce_result.num_hops)
        ablation_results['cce_spike'].append(cce_eval)

        print(f"    Retrieved: {[f.split('/')[-1] for f in cce_result.retrieved_files]}")
        print(f"    Hops: {cce_result.num_hops}, Recall: {mh_score['recall']:.2f}, Order: {mh_score['order_score']:.2f}")

        # === Random Baseline ===
        cleanup_memory()
        rand_result = target_random_baseline.retrieve(ex.task, num_retrievals=max(3, ex.min_hops_required), seed=global_idx)

        mh_score_rand = evaluate_multihop(rand_result.retrieved_files, ex)
        multihop_stats['random']['order_scores'].append(mh_score_rand['order_score'])
        multihop_stats['random']['hop_coverages'].append(mh_score_rand['hop_coverage'])
        if mh_score_rand['complete']:
            multihop_stats['random']['complete_count'] += 1

        if rand_result.retrieved_content:
            prompt = f"Context:\n{rand_result.retrieved_content[:3000]}\n\nTask: {ex.task}\nImplementation:"
        else:
            prompt = f"Task: {ex.task}\nImplementation:"

        rand_answer = generate_answer(prompt)
        rand_eval = evaluate_example(ex, rand_answer, rand_result.retrieved_files,
                                     len(tokenizer.encode(prompt)), 'random', rand_result.num_hops)
        ablation_results['random'].append(rand_eval)

        # === Fixed Interval ===
        cleanup_memory()
        fixed_result = target_fixed_baseline.retrieve(ex.task, interval=40, max_retrievals=max(3, ex.min_hops_required))

        mh_score_fixed = evaluate_multihop(fixed_result.retrieved_files, ex)
        multihop_stats['fixed']['order_scores'].append(mh_score_fixed['order_score'])
        multihop_stats['fixed']['hop_coverages'].append(mh_score_fixed['hop_coverage'])
        if mh_score_fixed['complete']:
            multihop_stats['fixed']['complete_count'] += 1

        if fixed_result.retrieved_content:
            prompt = f"Context:\n{fixed_result.retrieved_content[:3000]}\n\nTask: {ex.task}\nImplementation:"
        else:
            prompt = f"Task: {ex.task}\nImplementation:"

        fixed_answer = generate_answer(prompt)
        fixed_eval = evaluate_example(ex, fixed_answer, fixed_result.retrieved_files,
                                      len(tokenizer.encode(prompt)), 'fixed', fixed_result.num_hops)
        ablation_results['fixed'].append(fixed_eval)

        # === Query-Only (Single-hop) ===
        cleanup_memory()
        qo_results = target_retriever.retrieve(ex.task, top_k=3)
        qo_files = [r['source'] for r in qo_results]

        mh_score_qo = evaluate_multihop(qo_files, ex)
        multihop_stats['query_only']['order_scores'].append(mh_score_qo['order_score'])
        multihop_stats['query_only']['hop_coverages'].append(mh_score_qo['hop_coverage'])
        if mh_score_qo['complete']:
            multihop_stats['query_only']['complete_count'] += 1

        qo_context = "\n\n".join([r['content'][:1000] for r in qo_results])
        qo_prompt = f"Context:\n{qo_context}\n\nTask: {ex.task}\nImplementation:"
        qo_answer = generate_answer(qo_prompt)
        qo_eval = evaluate_example(ex, qo_answer, qo_files,
                                   len(tokenizer.encode(qo_prompt)), 'query_only', 1)
        ablation_results['query_only'].append(qo_eval)

        # === No Retrieval ===
        cleanup_memory()
        no_ret_prompt = f"Task: {ex.task}\nImplementation:"
        no_ret_answer = generate_answer(no_ret_prompt)
        no_ret_eval = evaluate_example(ex, no_ret_answer, [],
                                       len(tokenizer.encode(no_ret_prompt)), 'no_retrieval', 0)
        ablation_results['no_retrieval'].append(no_ret_eval)

        print(f"    CCE: {cce_eval.answer_correctness:.3f}, Query-Only: {qo_eval.answer_correctness:.3f}")

    cleanup_memory()

# =============================================================================
# RESULTS SUMMARY
# =============================================================================

print("\n" + "="*70)
print("ABLATION STUDY RESULTS (MULTI-HOP SCORING)")
print("="*70)

# Traditional metrics
for method in ['cce_spike', 'query_only', 'random', 'fixed', 'no_retrieval']:
    results = ablation_results[method]
    if results:
        avg_correct = sum(r.answer_correctness for r in results) / len(results)
        avg_recall = sum(r.context_recall for r in results) / len(results)
        avg_retrievals = sum(r.num_retrievals for r in results) / len(results)

        print(f"\n{method.upper()}:")
        print(f"  Answer Correctness: {avg_correct:.3f}")
        print(f"  Context Recall: {avg_recall:.3f}")
        print(f"  Avg Retrievals: {avg_retrievals:.2f}")

# Multi-hop specific metrics
print("\n" + "="*70)
print("MULTI-HOP METRICS (Query Evolution Validation)")
print("="*70)

for method in ['cce_spike', 'query_only', 'random', 'fixed']:
    stats = multihop_stats[method]
    if stats['order_scores']:
        avg_order = sum(stats['order_scores']) / len(stats['order_scores'])
        avg_coverage = sum(stats['hop_coverages']) / len(stats['hop_coverages'])
        complete_rate = stats['complete_count'] / len(stats['order_scores'])

        print(f"\n{method.upper()}:")
        print(f"  Order Score (retrieval sequence quality): {avg_order:.3f}")
        print(f"  Hop Coverage (% of required hops completed): {avg_coverage:.3f}")
        print(f"  Complete Rate (all required files): {complete_rate:.1%}")

# CCE-specific stats
print("\n" + "="*70)
print("CCE BEHAVIOR ANALYSIS")
print("="*70)
print(f"Total retrievals: {cce_stats['total_retrievals']}")
print(f"Avg retrievals per example: {cce_stats['total_retrievals']/len(sampled_examples):.2f}")
print(f"Query evolution uses (hop > 1): {cce_stats['query_evolution_used']}")


HTTPX ABLATION STUDY (MULTI-HOP SCORING)
Examples: 10
CCE Threshold: 2.5
Model: Qwen/Qwen2.5-Coder-1.5B-Instruct
Using 10 dynamic benchmark examples

Batch 1/4

[1/10] dyn_impl_001
  Task: Implement a retry mechanism for failed httpx requests with e...
  Expected: _client.py -> _exceptions.py -> _config.py
  Min hops: 3

  CCE-Spike:
    Token 0: CCE=0.483 (H_code=4.58, H_lang=4.09)
    Token 1: CCE=2.174 (H_code=3.48, H_lang=1.30)
    Token 2: CCE=2.282 (H_code=2.68, H_lang=0.40)
      Query (hop 1): original + confused
    SPIKE 1 at 15: CCE=3.83
      Query (hop 2): recent_ids=['self', 'None', 'pass', 'use', 'code'] + confused=['create', 'set', 'test', 'simulate', 'integrate']
    SPIKE 2 at 22: CCE=2.68
      Query (hop 3): recent_ids=['typing', 'Request', 'AsyncHandler', 'Coroutine', 'None'] + confused=['initial', 'import', 'slots', 'path', 'class']
    SPIKE 3 at 102: CCE=6.49
      Query (hop 4): recent_ids=['__aexit__', 'exc_value', 'code', 'from', 'http'] + confused=['client',

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


    Total: 5 retrievals, files: ['_exceptions.py', '_transports/base.py', '_urls.py', '_transports/mock.py', '_client.py', '__version__.py', '_config.py', '_utils.py']
    Retrieved: ['_exceptions.py', 'base.py', '_urls.py', 'mock.py', '_client.py', '__version__.py', '_config.py', '_utils.py']
    Hops: 5, Recall: 1.00, Order: 0.50
    CCE: 0.707, Query-Only: 0.510

[2/10] dyn_impl_002
  Task: Add request/response logging middleware to httpx client...
  Expected: _client.py -> base.py -> _models.py
  Min hops: 3

  CCE-Spike:
    Token 0: CCE=1.556 (H_code=5.76, H_lang=4.21)
    Token 1: CCE=0.733 (H_code=3.34, H_lang=2.61)
    Token 2: CCE=-0.895 (H_code=0.23, H_lang=1.13)
      Query (hop 1): original + confused
    SPIKE 1 at 11: CCE=3.29
      Query (hop 2): recent_ids=['creating', 'custom', 'class', 'and', 'instance'] + confused=['creating', 'middleware', 'implementation', 'this', 'custom']
    SPIKE 2 at 41: CCE=4.00
      Query (hop 3): recent_ids=['return', 'request', 'middlewa

In [19]:
# Cell E: Spike-Error Correlation (with error handling)

from scipy.stats import pearsonr, chi2_contingency # Added this line

print("="*70)
print("SPIKE-ERROR CORRELATION EXPERIMENT")
print("="*70)

# Verify token masks exist and have correct size
print("\nVerifying token masks...")
try:
    vocab_from_model = model.config.vocab_size
    mask_size = len(code_token_mask)
    print(f"Model vocab: {vocab_from_model}, Mask size: {mask_size}")

    if mask_size < vocab_from_model:
        print("WARNING: Mask smaller than vocab! Extending...")
        code_token_mask_new = np.zeros(vocab_from_model, dtype=bool)
        lang_token_mask_new = np.zeros(vocab_from_model, dtype=bool)
        code_token_mask_new[:mask_size] = code_token_mask
        lang_token_mask_new[:mask_size] = lang_token_mask
        code_token_mask = code_token_mask_new
        lang_token_mask = lang_token_mask_new
        print(f"Extended masks to size {len(code_token_mask)}")
except Exception as e:
    print(f"Token mask check error: {e}")

# Use subset of examples
analysis_examples = sampled_examples[:min(10, len(sampled_examples))]
print(f"\nAnalyzing {len(analysis_examples)} examples")

spike_data = []

for idx, ex in enumerate(analysis_examples):
    print(f"\n[{idx+1}/{len(analysis_examples)}] {ex.id}...")

    try:
        cleanup_memory()
        trace_result = generate_with_cce_trace(ex.task, max_tokens=80)

        print(f"  Tokens: {len(trace_result['tokens'])}, Spikes: {len(trace_result['spike_positions'])}")

        spike_data.append({
            'example_id': ex.id,
            'query': ex.task,
            'ground_truth': ex.ground_truth_answer,
            'tokens': trace_result['tokens'],
            'cce_trace': trace_result['cce_trace'],
            'spike_positions': trace_result['spike_positions'],
        })
    except Exception as e:
        print(f"  ERROR: {e}")
        continue

    cleanup_memory()

print(f"\nCollected {len(spike_data)} traces")

if len(spike_data) < 3:
    print("\nNot enough data for correlation analysis!")
    spike_verdict = "INSUFFICIENT DATA"
    criteria_met = 0
    best_threshold = SELECTED_THRESHOLD
    final_r_pearson = 0
    final_relative_risk = 0
    final_p_chi2 = 1
else:
    # Compute hallucination traces
    print("\nComputing hallucination traces...")

    for data in spike_data:
        try:
            hallu_trace = compute_hallucination_trace(data['tokens'], data['ground_truth'])
            data['hallu_trace'] = hallu_trace
            data['hallu_scores'] = [h['hallucination_score'] for h in hallu_trace]
            data['drop_positions'] = [i for i, h in enumerate(hallu_trace) if h.get('is_drop', False)]
            print(f"  {data['example_id']}: {len(data['drop_positions'])} drops")
        except Exception as e:
            print(f"  {data['example_id']}: Error - {e}")
            data['hallu_scores'] = [0] * len(data['tokens'])
            data['drop_positions'] = []

    # Multi-threshold analysis
    print("\n" + "="*70)
    print("MULTI-THRESHOLD ANALYSIS")
    print("="*70)

    thresholds_to_test = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
    threshold_results = []

    for thresh in thresholds_to_test:
        all_cce = []
        all_hallu = []
        all_spike_flags = []
        all_drop_flags = []

        for data in spike_data:
            cce_trace = data['cce_trace']
            hallu_scores = data.get('hallu_scores', [0] * len(cce_trace))
            drop_positions = data.get('drop_positions', [])
            min_len = min(len(cce_trace), len(hallu_scores))

            for i in range(min_len):
                all_cce.append(cce_trace[i])
                all_hallu.append(hallu_scores[i])
                all_spike_flags.append(1 if cce_trace[i] > thresh else 0)
                all_drop_flags.append(1 if i in drop_positions else 0)

        if len(all_cce) < 10:
            continue

        all_cce = np.array(all_cce)
        all_hallu = np.array(all_hallu)
        all_spike_flags = np.array(all_spike_flags)
        all_drop_flags = np.array(all_drop_flags)

        n_spikes = sum(all_spike_flags)

        # Correlation
        if len(all_cce) > 2 and np.std(all_cce) > 0 and np.std(all_hallu) > 0:
            r_pearson, p_pearson = pearsonr(all_cce, all_hallu)
        else:
            r_pearson, p_pearson = 0, 1

        # Conditional probability
        spike_and_drop = sum((all_spike_flags == 1) & (all_drop_flags == 1))
        spike_no_drop = sum((all_spike_flags == 1) & (all_drop_flags == 0))
        no_spike_drop = sum((all_spike_flags == 0) & (all_drop_flags == 1))
        no_spike_no_drop = sum((all_spike_flags == 0) & (all_drop_flags == 0))

        total_spikes = spike_and_drop + spike_no_drop
        total_no_spikes = no_spike_drop + no_spike_no_drop

        p_drop_spike = spike_and_drop / total_spikes if total_spikes > 0 else 0
        p_drop_no_spike = no_spike_drop / total_no_spikes if total_no_spikes > 0 else 0

        relative_risk = p_drop_spike / p_drop_no_spike if p_drop_no_spike > 0 else 0

        # Chi-square
        contingency = np.array([[spike_and_drop, spike_no_drop], [no_spike_drop, no_spike_no_drop]])
        if contingency.min() > 0:
            try:
                chi2, p_chi2, _, _ = chi2_contingency(contingency)
            except:
                chi2, p_chi2 = 0, 1
        else:
            chi2, p_chi2 = 0, 1

        threshold_results.append({
            'threshold': thresh,
            'n_spikes': n_spikes,
            'n_positions': len(all_cce),
            'r_pearson': r_pearson,
            'relative_risk': relative_risk,
            'p_chi2': p_chi2,
        })

        print(f"\nThreshold {thresh}: Spikes={n_spikes}, r={r_pearson:.3f}, RR={relative_risk:.2f}x, p={p_chi2:.3f}")

    # Find best threshold
    if threshold_results:
        # Best = highest relative risk that's > 1
        valid_results = [r for r in threshold_results if r['relative_risk'] > 0]
        if valid_results:
            best_result = max(valid_results, key=lambda x: x['relative_risk'])
        else:
            best_result = threshold_results[0]

        print("\n" + "="*70)
        print("BEST THRESHOLD")
        print("="*70)
        print(f"Threshold: {best_result['threshold']}")
        print(f"Relative Risk: {best_result['relative_risk']:.2f}x")
        print(f"Pearson r: {best_result['r_pearson']:.4f}")

        # Final verdict
        print("\n" + "="*70)
        print("FINAL VERDICT")
        print("="*70)

        criteria_met = 0
        if abs(best_result['r_pearson']) > 0.15:
            print(f"[PASS] |r|={abs(best_result['r_pearson']):.3f} > 0.15")
            criteria_met += 1
        else:
            print(f"[FAIL] |r|={abs(best_result['r_pearson']):.3f} < 0.15")

        if best_result['relative_risk'] > 1.1:
            print(f"[PASS] RR={best_result['relative_risk']:.2f}x > 1.1x")
            criteria_met += 1
        else:
            print(f"[FAIL] RR={best_result['relative_risk']:.2f}x < 1.1x")

        if best_result['p_chi2'] < 0.15:
            print(f"[PASS] p={best_result['p_chi2']:.3f} < 0.15")
            criteria_met += 1
        else:
            print(f"[FAIL] p={best_result['p_chi2']:.3f} >= 0.15")

        print(f"\nCriteria met: {criteria_met}/3")

        if criteria_met >= 2:
            spike_verdict = "SPIKES PREDICT ERRORS"
        elif criteria_met == 1:
            spike_verdict = "WEAK EVIDENCE"
        else:
            spike_verdict = "NO EVIDENCE"

        print(f"VERDICT: {spike_verdict}")

        best_threshold = best_result['threshold']
        final_r_pearson = best_result['r_pearson']
        final_relative_risk = best_result['relative_risk']
        final_p_chi2 = best_result['p_chi2']
    else:
        print("No valid threshold results!")
        spike_verdict = "ANALYSIS FAILED"
        criteria_met = 0
        best_threshold = SELECTED_THRESHOLD
        final_r_pearson = 0
        final_relative_risk = 0
        final_p_chi2 = 1


SPIKE-ERROR CORRELATION EXPERIMENT

Verifying token masks...
Model vocab: 151936, Mask size: 151936

Analyzing 10 examples

[1/10] dyn_impl_001...
  Tokens: 80, Spikes: 8

[2/10] dyn_impl_002...
  Tokens: 80, Spikes: 11

[3/10] dyn_debug_001...
  Tokens: 80, Spikes: 10

[4/10] dyn_debug_002...
  Tokens: 80, Spikes: 12

[5/10] dyn_ext_001...
  Tokens: 80, Spikes: 9

[6/10] dyn_ext_002...
  Tokens: 80, Spikes: 15

[7/10] dyn_int_001...
  Tokens: 80, Spikes: 11

[8/10] dyn_int_002...
  Tokens: 80, Spikes: 9

[9/10] dyn_complex_001...
  Tokens: 80, Spikes: 10

[10/10] dyn_impl_003...
  Tokens: 80, Spikes: 10

Collected 10 traces

Computing hallucination traces...
  dyn_impl_001: 4 drops
  dyn_impl_002: 8 drops
  dyn_debug_001: 10 drops
  dyn_debug_002: 5 drops
  dyn_ext_001: 11 drops
  dyn_ext_002: 4 drops
  dyn_int_001: 9 drops
  dyn_int_002: 10 drops
  dyn_complex_001: 9 drops
  dyn_impl_003: 6 drops

MULTI-THRESHOLD ANALYSIS

Threshold 1.5: Spikes=168, r=0.064, RR=1.00x, p=1.000

Thresh

In [20]:
# Cell F: Export Results (IMPROVED)

# Handle missing variables from previous cells
if 'threshold_results' not in dir():
    threshold_results = {}
    print('Note: threshold_results not available')
if 'best_threshold' not in dir():
    best_threshold = SELECTED_THRESHOLD
    print('Note: best_threshold not available, using SELECTED_THRESHOLD')
if 'correlation_data' not in dir():
    correlation_data = []

print("="*70)
print(f"{REPO_NAME.upper()} EXPERIMENT RESULTS")
print("="*70)

# Compute aggregate metrics
methods = ['cce_spike', 'random', 'fixed', 'query_only', 'no_retrieval']
agg_results = {}

for method in methods:
    results_list = ablation_results[method]
    if results_list:
        agg_results[method] = {
            'correctness': float(np.mean([r.answer_correctness for r in results_list])),
            'hallucination': float(np.mean([r.hallucination_rate for r in results_list])),
            'composite': float(np.mean([r.get_composite_score() for r in results_list])),
            'n': len(results_list)
        }

print("")
print("=== ABLATION RESULTS ===")
print(f"{'Method':<15} {'Correct':>10} {'Hallu':>10} {'Composite':>10}")
print("-" * 50)
for method in methods:
    if method in agg_results:
        m = agg_results[method]
        print(f"{method:<15} {m['correctness']:>10.3f} {m['hallucination']:>10.3f} {m['composite']:>10.3f}")

# Find best method
best_method = max(agg_results.keys(), key=lambda x: agg_results[x]['composite'])
print(f"\nBest method: {best_method} (composite={agg_results[best_method]['composite']:.3f})")

# CCE vs baselines
cce_comp = agg_results['cce_spike']['composite']
rand_comp = agg_results['random']['composite']
fixed_comp = agg_results['fixed']['composite']

print("")
print("=== CCE vs BASELINES ===")
print(f"CCE Composite:    {cce_comp:.3f}")
print(f"Random Composite: {rand_comp:.3f}")
print(f"Fixed Composite:  {fixed_comp:.3f}")
print(f"CCE - Random:     {cce_comp - rand_comp:+.3f}")
print(f"CCE - Fixed:      {cce_comp - fixed_comp:+.3f}")

if cce_comp > rand_comp + 0.03:
    ablation_verdict = "CCE OUTPERFORMS RANDOM"
elif cce_comp < rand_comp - 0.03:
    ablation_verdict = "RANDOM OUTPERFORMS CCE"
else:
    ablation_verdict = "NO SIGNIFICANT DIFFERENCE"

print(f"Verdict: {ablation_verdict}")

print("")
print("=== SPIKE-ERROR CORRELATION ===")
print(f"Best threshold:  {best_threshold}")
print(f"Pearson r:       {final_r_pearson:.4f}")
print(f"Relative Risk:   {final_relative_risk:.2f}x")
print(f"Chi-square p:    {final_p_chi2:.4f}")
print(f"Criteria met:    {criteria_met}/3")
print(f"Verdict:         {spike_verdict}")

# Custom encoder for NumPy types
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

# Export
export_data = {
    'experiment': f'{REPO_NAME}_cce_ablation_v2',
    'config': {
        'model': MODEL_NAME,
        'codebase': REPO_NAME,
        'cce_threshold': SELECTED_THRESHOLD,
        'best_threshold': best_threshold,
        'num_examples': len(sampled_examples),
    },
    'ablation': agg_results,
    'threshold_analysis': threshold_results,
    'spike_error': {
        'best_threshold': best_threshold,
        'pearson_r': float(final_r_pearson),
        'relative_risk': float(final_relative_risk) if final_relative_risk != float('inf') else 999,
        'chi2_p': float(final_p_chi2),
        'criteria_met': criteria_met,
        'verdict': spike_verdict,
    },
    'overall_verdict': ablation_verdict,
}

output_file = f'{REPO_NAME}_results_v2.json'
with open(output_file, 'w') as f:
    json.dump(export_data, f, indent=2, cls=NpEncoder)

print("")
print(f"Exported: {output_file}")
print("")
print("="*70)
print("EXPERIMENT COMPLETE")
print("="*70)

HTTPX EXPERIMENT RESULTS

=== ABLATION RESULTS ===
Method             Correct      Hallu  Composite
--------------------------------------------------
cce_spike            0.537      0.146      0.600
random               0.485      0.252      0.535
fixed                0.511      0.062      0.587
query_only           0.515      0.350      0.551
no_retrieval         0.516      0.200      0.523

Best method: cce_spike (composite=0.600)

=== CCE vs BASELINES ===
CCE Composite:    0.600
Random Composite: 0.535
Fixed Composite:  0.587
CCE - Random:     +0.065
CCE - Fixed:      +0.013
Verdict: CCE OUTPERFORMS RANDOM

=== SPIKE-ERROR CORRELATION ===
Best threshold:  3.0
Pearson r:       0.0638
Relative Risk:   1.24x
Chi-square p:    0.6577
Criteria met:    1/3
Verdict:         WEAK EVIDENCE

Exported: httpx_results_v2.json

EXPERIMENT COMPLETE
